# DOFBOT Forward Kinematics & Sensitivity Lab
**Robotics I · Team 12 · Part 1 + 6000-level extension**

This notebook computes a complete end-effector pose from joint angles, provides **five distinct test sets**, compares physical measurements with predictions, and studies small changes to each joint using nonlinear forward kinematics and a geometric Jacobian.

**Start here:** Run all cells in Google Colab or Jupyter with Python 3. The default run computes predictions and creates blank measurement CSVs; it sends **no motor commands**. Replace the illustrative geometry/calibration, collect measurements on your arm, then rerun the analysis. Blank measurements remain missing, never synthetic experimental results.

The model treats S1–S5 as positioning joints and S6 as the gripper opening. S6 is held fixed because opening the fingers is not a sixth revolute joint of the rigid tool pose. This is the common DOFBOT layout; verify your model and assembly.

| Assignment item | Notebook section |
|---|---|
| Pose function from joint angles | 2: `forward_kinematics(q_rad)` |
| At least five configurations | 3: test plan |
| Measure each end-effector pose | 4: measurement protocol and editable CSV |
| Compare measured and predicted poses | 5: position and rotation errors |
| Individual small joint perturbations | 6: ±1° and ±2° trials |
| Compare measured changes and FK changes | 7: sensitivity tables |
| Relate observations to Jacobian | 6–8: analytic Jacobian, numerical check, empirical columns |


## 1. Setup, units, and editable hardware parameters
All geometry and measured positions use **millimeters**. Kinematic functions use **radians**. Servo commands and measurement roll/pitch/yaw use **degrees**. Predicted poses map tool coordinates into the fixed base frame: `p_base = R_base_tool @ p_tool + t_base_tool`.

Cloud Colab cannot directly access the arm's local I²C/serial controller. Use the vendor app/onboard Jupyter to position the arm and enter measurements here. The optional control helpers at the end can run on the DOFBOT's onboard computer with its existing vendor `Arm_Lib`; the analysis has no dependency on that library.


In [ ]:
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj): print(obj)
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("Plotting unavailable. In Colab these libraries are preinstalled; locally install matplotlib for figures.")
np.set_printoptions(precision=5, suppress=True)
pd.set_option("display.max_columns", 30)
OUTPUT_DIR = Path("dofbot_lab_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Python analysis ready. Output folder:", OUTPUT_DIR.resolve())


### Define the zero geometry before interpreting predictions
**Numbers below are illustrative placeholders, not verified DOFBOT dimensions.** Measure pivot-center distances on your actual arm or obtain its exact URDF/CAD. In the model's `q=0` configuration:

- Base origin is the center of S1 at the base datum; +Z points upward. +X points along the straight reference arm; +Y completes the right-handed frame.
- S1 rotates about +Z. S2, S3, S4 rotate about +Y (positive rotation turns +X toward −Z). S5 rolls about +X along the straight wrist.
- The shoulder is `base_height_mm` above the datum. Upper arm, forearm, and wrist offsets extend along +X. Tool origin is a marked point on the rigid gripper housing or a fixed tip with the fingers held constant.
- `tool_offset_mm` is measured from S5's pivot to that marked point in the zero configuration. An off-axis point moves when S5 rolls. Tool orientation at zero is `R_TOOL_HOME`.

`JOINT_POINTS_HOME` and `AXES_HOME` describe joint centers and unit axes in the **same base frame at q=0**. Edit them directly if your assembly has lateral offsets or different axes; the kinematic engine and Jacobian still apply. The zero geometry need not be a physically reachable pose, but must match the geometry used for calibration.

Measure dimensions independently of the five validation cases. Use separate calibration poses to estimate zeros/signs, then freeze parameters before collecting validation data.


In [ ]:
# EDIT these geometry parameters using your arm's pivot centers, in mm.
GEOMETRY = dict(base_height_mm=60.0, upper_arm_mm=80.0,
                forearm_mm=80.0, wrist_offset_mm=35.0)
TOOL_OFFSET_MM = np.array([55.0, 0.0, 0.0])
R_TOOL_HOME = np.eye(3)
GEOMETRY_VERIFIED = False  # set True only after measuring/confirming your geometry

h, a, b, c = [GEOMETRY[k] for k in
               ("base_height_mm", "upper_arm_mm", "forearm_mm", "wrist_offset_mm")]
JOINT_POINTS_HOME = np.array([[0,0,0], [0,0,h], [a,0,h],
                              [a+b,0,h], [a+b+c,0,h]], dtype=float)
AXES_HOME = np.array([[0,0,1], [0,1,0], [0,1,0], [0,1,0], [1,0,0]], dtype=float)
M_HOME = np.eye(4)
M_HOME[:3,:3] = R_TOOL_HOME
M_HOME[:3,3] = JOINT_POINTS_HOME[-1] + TOOL_OFFSET_MM

# EDIT: map PUBLIC Arm_Lib servo angles to model joint angles.
# q = radians(s - SERVO_ZERO_DEG) * SERVO_GAIN + JOINT_OFFSET_RAD
# These zeros, signs, and gains are examples, NOT manufacturer calibration.
SERVO_ZERO_DEG = np.array([90., 90., 90., 90., 135.])
SERVO_GAIN = np.array([1., -1., -1., -1., 1.])
JOINT_OFFSET_RAD = np.zeros(5)
CALIBRATION_VERIFIED = False
JOINT_NAMES = ["base", "shoulder", "elbow", "wrist_pitch", "wrist_roll"]
# Public SDK ranges; restrict these further for your assembly/collision limits.
SERVO_LIMITS_DEG = np.array([[0,180], [0,180], [0,180], [0,180], [0,270]], float)
GRIPPER_DEG = 90  # fixed S6 angle; choose a suitable setting for your tool point

assert np.all(np.isfinite(JOINT_POINTS_HOME))
assert np.allclose(np.linalg.norm(AXES_HOME, axis=1), 1)
assert np.allclose(R_TOOL_HOME.T @ R_TOOL_HOME, np.eye(3)) and np.linalg.det(R_TOOL_HOME) > 0
assert np.all(np.isfinite(SERVO_GAIN)) and np.all(SERVO_GAIN != 0)
if not (GEOMETRY_VERIFIED and CALIBRATION_VERIFIED):
    print("ILLUSTRATIVE MODEL: edit and verify geometry/calibration before hardware conclusions.")
display(pd.DataFrame({"joint":JOINT_NAMES, "servo_zero_deg":SERVO_ZERO_DEG,
                      "signed_gain":SERVO_GAIN}))


### Calibrate servo conventions
Use **public SDK angles**, not raw pulse widths. Record a known physical configuration, define its model angles, and solve the zero offsets. For each joint, change the public command by a small positive amount and observe whether rotation agrees with the right-hand direction above; set the sign of `SERVO_GAIN` accordingly. A fitted gain magnitude can correct angular scale. Do not add a second firmware/pulse inversion just because a motor is mounted oppositely.

For measured reference angles `q_ref`, known commands `s_ref`, and chosen gains:
`SERVO_ZERO_DEG = s_ref - degrees((q_ref - JOINT_OFFSET_RAD) / SERVO_GAIN)`.
Verify predictions at independent poses after calibration. Servo feedback is useful for checking settling, but it is not an independent measurement of end-effector pose.


In [ ]:
def vector5(x, name="joint vector"):
    x = np.asarray(x, dtype=float)
    if x.shape != (5,) or not np.isfinite(x).all():
        raise ValueError(f"{name} must have five finite values.")
    return x

def check_servo_limits(s_deg):
    s = vector5(s_deg, "servo angles")
    if np.any(s < SERVO_LIMITS_DEG[:,0]) or np.any(s > SERVO_LIMITS_DEG[:,1]):
        raise ValueError("Servo angle outside configured limits.")
    return s

def servo_to_q(s_deg):
    return np.deg2rad(check_servo_limits(s_deg) - SERVO_ZERO_DEG) * SERVO_GAIN + JOINT_OFFSET_RAD

def q_to_servo(q_rad):
    return check_servo_limits(SERVO_ZERO_DEG + np.rad2deg(
        (vector5(q_rad) - JOINT_OFFSET_RAD) / SERVO_GAIN))


## 2. Forward kinematics: complete pose
This is a product-of-exponentials formulation of the serial chain. For a revolute joint with home unit axis $a_i$ through point $o_i$, rotation about that fixed line is
$$E_i(q_i)=\begin{bmatrix}R(a_i,q_i)&(I-R(a_i,q_i))o_i\\0&1\end{bmatrix},\qquad
T(q)=E_1(q_1)E_2(q_2)\cdots E_5(q_5)M.$$
The upper-left block of `T` is orientation, the upper-right column is position. This supports explicit physical axes without guessing a DH table. Joint axes for later joints are transported by preceding rotations.

For display and entry, roll/pitch/yaw means **fixed-axis X then Y then Z**, so $R=R_z(yaw)R_y(pitch)R_x(roll)$. Euler coordinates are ambiguous at pitch ±90°; matrix/rotation-vector errors below remain valid there. Orientation differences are never calculated by subtracting Euler angles.


In [ ]:
def skew(v):
    x,y,z = np.asarray(v, float)
    return np.array([[0,-z,y],[z,0,-x],[-y,x,0]])

def rotation_axis(axis, angle):
    K = skew(axis)
    return np.eye(3) + np.sin(angle)*K + (1-np.cos(angle))*(K @ K)

def joint_transform(axis, point, angle):
    T = np.eye(4)
    T[:3,:3] = rotation_axis(axis, angle)
    T[:3,3] = (np.eye(3) - T[:3,:3]) @ point
    return T

def forward_kinematics(q_rad):
    """Return 4x4 base-to-tool pose; input five model angles in radians."""
    q = vector5(q_rad)
    T = np.eye(4)
    for axis, point, angle in zip(AXES_HOME, JOINT_POINTS_HOME, q):
        T = T @ joint_transform(axis, point, angle)
    return T @ M_HOME

def fk_servo(s_deg):
    """Convenience wrapper for five public Arm_Lib angles in degrees."""
    return forward_kinematics(servo_to_q(s_deg))

def rpy_to_R(rpy_deg):
    r,p,y = np.deg2rad(np.asarray(rpy_deg, float))
    return (rotation_axis([0,0,1],y) @ rotation_axis([0,1,0],p)
            @ rotation_axis([1,0,0],r))

def R_to_rpy(R):
    cp = np.hypot(R[0,0], R[1,0])
    pitch = np.arctan2(-R[2,0], cp)
    if cp > 1e-9:
        roll, yaw = np.arctan2(R[2,1],R[2,2]), np.arctan2(R[1,0],R[0,0])
    else:
        # Choose roll=0 at gimbal lock; yaw absorbs the ambiguity.
        roll, yaw = 0., np.arctan2(-R[0,1],R[1,1])
    return np.rad2deg([roll,pitch,yaw])

def rotation_log(R):
    """SO(3) logarithm vector, in radians; robust near zero and pi."""
    theta = np.arccos(np.clip((np.trace(R)-1)/2, -1., 1.))
    vee = np.array([R[2,1]-R[1,2], R[0,2]-R[2,0], R[1,0]-R[0,1]])
    if theta < 1e-7:
        return vee / 2
    if np.pi-theta < 1e-6:
        vals, vecs = np.linalg.eigh((R + R.T + 2*np.eye(3))/4)
        axis = vecs[:,np.argmax(vals)]
        if np.dot(axis,vee) < 0: axis = -axis
        return theta*axis
    return theta/(2*np.sin(theta))*vee

def pose_record(T):
    return dict(zip(["x_mm","y_mm","z_mm","roll_deg","pitch_deg","yaw_deg"],
                    np.r_[T[:3,3], R_to_rpy(T[:3,:3])]))

example_s = np.array([90,120,70,80,135], float)
print("Example model joint angles (deg):", np.rad2deg(servo_to_q(example_s)))
print("T_base_tool =\n", fk_servo(example_s))
display(pd.DataFrame([pose_record(fk_servo(example_s))]))


## 3. Five joint-angle test sets
The following public servo commands are **editable candidate configurations**, not collision-checked trajectories. After calibration, choose five reachable, distinct configurations that vary base, shoulder, elbow, wrist pitch, and wrist roll. Keep the same marked tool point and gripper opening. Check travel paths on your actual arm.

Record the command and, if available, the settled servo readback. The `s*_deg` columns in the measurement sheet are the angles used for prediction: initially the commands, but you may replace them with settled feedback and explain this choice. Do not change test IDs.


In [ ]:
SERVO_COLUMNS = [f"s{i}_deg" for i in range(1,6)]
POSE_COLUMNS = ["x_mm","y_mm","z_mm","roll_deg","pitch_deg","yaw_deg"]
test_sets = np.array([[90,120,70,80,135], [65,110,85,95,135],
                      [115,130,60,90,135], [80,100,105,70,110],
                      [105,115,80,110,165]], float)
test_plan = pd.DataFrame(test_sets, columns=SERVO_COLUMNS)
test_plan.insert(0,"test_id", [f"T{i}" for i in range(1,len(test_sets)+1)])
for s in test_sets: check_servo_limits(s)
predicted = pd.DataFrame([{"test_id":row.test_id, **pose_record(fk_servo(
    row[SERVO_COLUMNS].to_numpy(float)))} for _,row in test_plan.iterrows()])
display(test_plan.merge(predicted, on="test_id"))
test_plan.to_csv(OUTPUT_DIR/"five_test_plan.csv", index=False)


## 4. Collect physical end-effector measurements
1. Mark the tool origin and establish the base datum and +X/+Y/+Z axes. Fix the arm base and hold S6 constant. Use a ruler/caliper, calibrated camera, or tracked marker to measure the marked point in mm.
2. For each test, command the pose, wait for settling, record the angles used for prediction, and measure X/Y/Z. Take at least three repeats if practical and preserve the raw readings; enter their mean here and record repeat count/uncertainty in `notes`.
3. Measure orientation independently. A tracked rigid marker or three noncollinear points defines a full tool frame. A single point or tool direction alone does **not** measure roll. Convert the measured frame to the stated roll/pitch/yaw convention; an inclinometer may only measure some components.
4. If orientation is not measured, leave all three angle cells blank. Position comparisons still run, but the full-pose requirement remains incomplete. Do not fill orientation from FK.
5. Record instrument, position/orientation uncertainty, approach direction, load, settling time, and whether angles are commands or feedback. Reapproach the same way to control backlash, and test both directions separately if investigating it.

If a tracker uses a world frame W, convert all observations to this base frame B using a measured fixed transform: $T^B_E=(T^W_B)^{-1}T^W_E$. The tracked marker-to-tool offset must also be included. Never compare poses in different frames.

**Data entry:** Run the next cell to create a blank sheet. Edit `fk_measurements` in a new cell, or fill `fk_measurements_template.csv` externally and set `FK_CSV_PATH` to its path. Example syntax: `fk_measurements.loc[0, ['x_mm','y_mm','z_mm']] = [your_x, your_y, your_z]`. Re-run sections 5 onward after entry. Template files are only created when absent, so repeated runs do not erase an edited CSV.


In [ ]:
fk_template = test_plan.copy()
for c in POSE_COLUMNS: fk_template[c] = np.nan
fk_template["notes"] = ""
FK_CSV_PATH = None  # e.g. "/content/fk_measurements_completed.csv"
if FK_CSV_PATH:
    fk_measurements = pd.read_csv(FK_CSV_PATH)
else:
    fk_measurements = fk_template.copy()
template_path = OUTPUT_DIR/"fk_measurements_template.csv"
if not template_path.exists(): fk_template.to_csv(template_path, index=False)
display(fk_measurements)

def validate_measurements(df, expected_ids, id_col):
    required = [id_col] + SERVO_COLUMNS + POSE_COLUMNS
    if set(required)-set(df.columns): raise ValueError("Missing columns: " + str(set(required)-set(df.columns)))
    if df[id_col].duplicated().any() or set(df[id_col]) != set(expected_ids):
        raise ValueError("Use exactly the planned IDs, once each.")
    for c in SERVO_COLUMNS + POSE_COLUMNS:
        df[c] = pd.to_numeric(df[c], errors="raise")
        if np.isinf(df[c].to_numpy(float)).any(): raise ValueError("Infinite value in " + c)
    for _,r in df.iterrows(): check_servo_limits(r[SERVO_COLUMNS].to_numpy(float))
    for group in (POSE_COLUMNS[:3], POSE_COLUMNS[3:]):
        counts = df[group].notna().sum(axis=1)
        if ((counts != 0) & (counts != 3)).any():
            raise ValueError("Fill all three position or orientation components, or leave that group blank.")
    return df

def measured_pose(row):
    pvals = row[POSE_COLUMNS[:3]].to_numpy(float)
    avals = row[POSE_COLUMNS[3:]].to_numpy(float)
    return (pvals if np.isfinite(pvals).all() else None,
            rpy_to_R(avals) if np.isfinite(avals).all() else None)


## 5. Measured-versus-predicted pose comparison
Position error is $e_p=p_{measured}-p_{predicted}$ in mm. Its norm is the Euclidean distance error. Orientation error is the shortest rotation between the two frames:
$$e_R=\log(R_{measured}R_{predicted}^T)^\vee,\qquad \theta_{error}=\|e_R\|.$$
The rotation vector is expressed in the base frame. Position RMSE below is $\sqrt{\operatorname{mean}(\|e_p\|^2)}$; each coordinate's RMSE is also reported. Missing data do not contribute to summaries.


In [ ]:
validate_measurements(fk_measurements, test_plan.test_id, "test_id")
comparison_rows = []
for _,r in fk_measurements.iterrows():
    T = fk_servo(r[SERVO_COLUMNS].to_numpy(float))
    pm,Rm = measured_pose(r)
    ep = pm-T[:3,3] if pm is not None else np.full(3,np.nan)
    er = rotation_log(Rm @ T[:3,:3].T) if Rm is not None else np.full(3,np.nan)
    out = {"test_id":r.test_id}
    out.update({"pred_"+k:v for k,v in pose_record(T).items()})
    out.update({"meas_"+k:r[k] for k in POSE_COLUMNS})
    out.update(dict(zip(["error_x_mm","error_y_mm","error_z_mm"],ep)))
    out["position_error_mm"] = np.linalg.norm(ep)
    out["orientation_error_deg"] = np.rad2deg(np.linalg.norm(er))
    comparison_rows.append(out)
fk_comparison = pd.DataFrame(comparison_rows)
display(fk_comparison)
valid_p = fk_comparison.dropna(subset=["position_error_mm"])
valid_R = fk_comparison.dropna(subset=["orientation_error_deg"])
print(f"Measured position: {len(valid_p)}/5; measured orientation: {len(valid_R)}/5")
if len(valid_p):
    e = valid_p[["error_x_mm","error_y_mm","error_z_mm"]].to_numpy()
    print("Position RMSE (mm):", np.sqrt(np.mean(np.sum(e**2,axis=1))))
    print("Coordinate RMSE X,Y,Z (mm):", np.sqrt(np.mean(e**2,axis=0)))
    print("Mean / maximum position error (mm):", valid_p.position_error_mm.mean(), valid_p.position_error_mm.max())
if len(valid_R):
    print("Orientation RMSE (deg):", np.sqrt(np.mean(valid_R.orientation_error_deg**2)))
if not len(valid_p): print("Enter physical measurements to calculate experimental errors.")


In [ ]:
if plt is not None:
    fig = plt.figure(figsize=(11,4))
    ax = fig.add_subplot(121, projection="3d")
    ax.scatter(fk_comparison.pred_x_mm, fk_comparison.pred_y_mm, fk_comparison.pred_z_mm, label="Predicted")
    if len(valid_p):
        ax.scatter(valid_p.meas_x_mm, valid_p.meas_y_mm, valid_p.meas_z_mm, marker="x", label="Measured")
        for _,r in valid_p.iterrows():
            ax.plot([r.pred_x_mm,r.meas_x_mm],[r.pred_y_mm,r.meas_y_mm],[r.pred_z_mm,r.meas_z_mm],"k--",alpha=.5)
    ax.set(xlabel="X (mm)",ylabel="Y (mm)",zlabel="Z (mm)",title="Five pose tests")
    ax.legend()
    ax2 = fig.add_subplot(122)
    if len(valid_p):
        ax2.bar(valid_p.test_id, valid_p.position_error_mm)
        ax2.set(xlabel="Test",ylabel="Position error (mm)",title="Physical validation")
    else:
        ax2.text(.5,.5,"Awaiting physical measurements",ha="center",va="center",transform=ax2.transAxes)
        ax2.set_axis_off()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/"fk_comparison.png",dpi=160)
    plt.show()


## 6. 6000-level sensitivity: individual perturbations and Jacobian
Select a well-conditioned, collision-free baseline configuration. This plan uses T1 and changes only **one public servo command** at a time by ±1° and ±2°. Calibration maps the perturbation into model radians: $\Delta q_i=g_i\Delta s_i$ (both angular changes in radians).

For current joint axis $a_i(q)$ through $o_i(q)$ and tool point $p(q)$, the geometric Jacobian is
$$J(q)=\begin{bmatrix}J_p\\J_\omega\end{bmatrix},\quad
J_{p,i}=a_i\times(p-o_i),\quad J_{\omega,i}=a_i.$$
It maps model joint rates to **tool-origin linear velocity and base-frame angular velocity**. It is not an Euler-angle-rate Jacobian. Thus $\Delta p\approx J_p\Delta q$ and $\log(R(q+\Delta q)R(q)^T)^\vee\approx J_\omega\Delta q$.

When perturbations are in public servo radians, use $J_s=J\,\operatorname{diag}(g)$. Translational rows have units mm/rad; rotational rows rad/rad. A 6×5 Jacobian cannot span arbitrary six-dimensional pose changes. Inspect position singular values separately because mixed mm/rad and rad/rad rows need deliberate scaling before interpreting a combined condition number.


In [ ]:
def current_joint_axes(q_rad):
    prefix = np.eye(4)
    axes, origins = [], []
    for a,o,q in zip(AXES_HOME, JOINT_POINTS_HOME, vector5(q_rad)):
        axes.append(prefix[:3,:3] @ a)
        origins.append(prefix[:3,:3] @ o + prefix[:3,3])
        prefix = prefix @ joint_transform(a,o,q)
    return np.array(axes), np.array(origins)

def geometric_jacobian(q_rad):
    p = forward_kinematics(q_rad)[:3,3]
    axes, origins = current_joint_axes(q_rad)
    return np.vstack([np.cross(axes,p-origins).T, axes.T])

def numerical_jacobian(q_rad, epsilon=1e-6):
    q = vector5(q_rad)
    J = np.zeros((6,5))
    for i in range(5):
        dq = np.zeros(5); dq[i] = epsilon
        Tp,Tm = forward_kinematics(q+dq), forward_kinematics(q-dq)
        J[:3,i] = (Tp[:3,3]-Tm[:3,3])/(2*epsilon)
        J[3:,i] = rotation_log(Tp[:3,:3] @ Tm[:3,:3].T)/(2*epsilon)
    return J

BASELINE_SERVO_DEG = test_sets[0].copy()  # EDIT to select your baseline
PERTURBATIONS_DEG = [1.0, 2.0]  # smaller steps may be below servo/instrument resolution
q0 = servo_to_q(BASELINE_SERVO_DEG)
T0 = forward_kinematics(q0)
J0 = geometric_jacobian(q0)
Jservo = J0 @ np.diag(SERVO_GAIN)
display(pd.DataFrame(J0, index=["vx_mm/rad","vy_mm/rad","vz_mm/rad",
                             "wx_rad/rad","wy_rad/rad","wz_rad/rad"], columns=JOINT_NAMES))
print("Position singular values (mm/rad):", np.linalg.svd(J0[:3],compute_uv=False))
print("Position rank:", np.linalg.matrix_rank(J0[:3]), "; full geometric rank:", np.linalg.matrix_rank(J0))


### Internal numerical checks
These verify the kinematic implementation and Jacobian, **not the physical DOFBOT calibration**. The geometric Jacobian must match independent centered differences of FK at several nonzero configurations. Rotations must be proper and homogeneous transforms valid.


In [ ]:
assert np.allclose(forward_kinematics(np.zeros(5)),M_HOME,atol=1e-10)
for s in test_sets:
    q = servo_to_q(s)
    T = forward_kinematics(q)
    assert np.allclose(T[3],[0,0,0,1])
    assert np.allclose(T[:3,:3].T @ T[:3,:3],np.eye(3),atol=1e-10)
    assert np.isclose(np.linalg.det(T[:3,:3]),1)
    assert np.allclose(q_to_servo(q),s)
    assert np.allclose(geometric_jacobian(q),numerical_jacobian(q),atol=2e-5,rtol=1e-6)
print("Passed home-pose, rotation, calibration round-trip, and analytic/numerical Jacobian checks at five poses.")


In [ ]:
trial_rows = [{"trial_id":"baseline", "joint_id":0, "delta_servo_deg":0.,
               **dict(zip(SERVO_COLUMNS,BASELINE_SERVO_DEG))}]
for i in range(5):
    for magnitude in PERTURBATIONS_DEG:
        for sign in [-1,1]:
            delta = sign*magnitude
            s = BASELINE_SERVO_DEG.copy(); s[i] += delta
            check_servo_limits(s)
            trial_rows.append({"trial_id":f"J{i+1}_{'plus' if sign>0 else 'minus'}_{magnitude:g}",
                               "joint_id":i+1,"delta_servo_deg":delta,
                               **dict(zip(SERVO_COLUMNS,s))})
sensitivity_plan = pd.DataFrame(trial_rows)
sensitivity_template = sensitivity_plan.copy()
for c in POSE_COLUMNS: sensitivity_template[c] = np.nan
sensitivity_template["notes"] = ""
SENSITIVITY_CSV_PATH = None  # e.g. "/content/sensitivity_measurements_completed.csv"
sensitivity_measurements = (pd.read_csv(SENSITIVITY_CSV_PATH) if SENSITIVITY_CSV_PATH
                            else sensitivity_template.copy())
p = OUTPUT_DIR/"sensitivity_measurements_template.csv"
if not p.exists(): sensitivity_template.to_csv(p,index=False)
sensitivity_plan.to_csv(OUTPUT_DIR/"sensitivity_plan.csv",index=False)
display(sensitivity_plan)


## 7. Measure changes and compare nonlinear FK with the Jacobian
Measure the baseline pose, then each perturbed pose. Return to the baseline between trials and verify it has not drifted. Keep other joints, gripper opening, tool point, load, and measurement frame fixed. Repeat and log uncertainty/backlash. If drift matters, record a separate baseline immediately before each trial in your raw lab log and adapt the comparison accordingly; the sheet below assumes one stable baseline.

The sheet has **21 rows: one baseline and 20 perturbations**. Fill the measured pose columns just as in Part 1. Preserve the planned servo angle columns for centered single-joint sensitivity; store readback separately in `notes`/your raw log. If using actual angular readback, refit an empirical derivative using the actual angle separation rather than silently using the nominal denominator.

For each trial, the table compares:
- Nonlinear FK changes: $p(q+\Delta q)-p(q)$ and the base-frame relative rotation vector.
- Linear Jacobian changes: $J_p\Delta q$ and $J_\omega\Delta q$.
- Physical changes: measured perturbed position/orientation relative to the measured baseline.

The measured residual subtracts changes, so a constant position datum offset cancels. A frame rotation, calibration error, or varying deflection still matters. Rotation changes below are vectors in radians, converted to degrees for display; do not interpret them as roll/pitch/yaw changes.


In [ ]:
validate_measurements(sensitivity_measurements, sensitivity_plan.trial_id, "trial_id")
expected = sensitivity_plan.set_index("trial_id")
for _,r in sensitivity_measurements.iterrows():
    if not np.allclose(r[SERVO_COLUMNS].to_numpy(float), expected.loc[r.trial_id,SERVO_COLUMNS].to_numpy(float)):
        raise ValueError("Sensitivity servo columns must match the planned single-joint commands.")
base_row = sensitivity_measurements.set_index("trial_id").loc["baseline"]
pb,Rb = measured_pose(base_row)
sensitivity_rows = []
for _,r in sensitivity_measurements.iterrows():
    if r.trial_id == "baseline": continue
    plan = expected.loc[r.trial_id]
    q = servo_to_q(r[SERVO_COLUMNS].to_numpy(float)); dq = q-q0
    T = forward_kinematics(q)
    dp_fk = T[:3,3]-T0[:3,3]
    dw_fk = rotation_log(T[:3,:3] @ T0[:3,:3].T)
    dp_J,dw_J = (J0 @ dq)[:3], (J0 @ dq)[3:]
    pm,Rm = measured_pose(r)
    dp_m = pm-pb if pm is not None and pb is not None else np.full(3,np.nan)
    dw_m = rotation_log(Rm @ Rb.T) if Rm is not None and Rb is not None else np.full(3,np.nan)
    out = {"trial_id":r.trial_id, "joint_id":int(plan.joint_id),
           "delta_servo_deg":plan.delta_servo_deg,
           "delta_q_deg":np.rad2deg(dq[int(plan.joint_id)-1])}
    for prefix,v in [("fk_dp",dp_fk),("J_dp",dp_J),("meas_dp",dp_m)]:
        out.update({f"{prefix}_{k}_mm":x for k,x in zip("xyz",v)})
    for prefix,v in [("fk_dw",dw_fk),("J_dw",dw_J),("meas_dw",dw_m)]:
        out.update({f"{prefix}_{k}_deg":x for k,x in zip("xyz",np.rad2deg(v))})
    out.update(linearization_position_error_mm=np.linalg.norm(dp_fk-dp_J),
               linearization_rotation_vector_error_deg=np.rad2deg(np.linalg.norm(dw_fk-dw_J)),
               measured_vs_fk_position_error_mm=np.linalg.norm(dp_m-dp_fk),
               measured_vs_J_position_error_mm=np.linalg.norm(dp_m-dp_J),
               measured_vs_fk_rotation_vector_error_deg=np.rad2deg(np.linalg.norm(dw_m-dw_fk)),
               measured_vs_J_rotation_vector_error_deg=np.rad2deg(np.linalg.norm(dw_m-dw_J)))
    sensitivity_rows.append(out)
sensitivity_comparison = pd.DataFrame(sensitivity_rows)
display(sensitivity_comparison[["trial_id","delta_q_deg","linearization_position_error_mm",
    "linearization_rotation_vector_error_deg","measured_vs_fk_position_error_mm",
    "measured_vs_J_position_error_mm","measured_vs_fk_rotation_vector_error_deg",
    "measured_vs_J_rotation_vector_error_deg"]])
print("Complete measured changes:", sensitivity_comparison.measured_vs_fk_position_error_mm.notna().sum(),
      "position trials;", sensitivity_comparison.measured_vs_fk_rotation_vector_error_deg.notna().sum(),"orientation trials.")


### Estimate Jacobian columns from physical centered differences
For a single servo perturbation of $h$ radians, an empirical column is
$$\widehat J_{s,p,i}=\frac{p_{+h}-p_{-h}}{2h},\qquad
\widehat J_{s,\omega,i}=\frac{\log(R_{+h}R_{-h}^T)^\vee}{2h}.$$
Compare these with `Jservo[:, i]`. The derivative is with respect to **servo radians**; divide by the signed gain to express it per model radian. Centered differencing suppresses first-order truncation bias, but measurement noise is amplified by division by a small `h`. For position measurement standard deviation $\sigma_p$, independent ± measurements give component derivative uncertainty approximately $\sigma_p/(\sqrt{2}h)$. Backlash and drift need not be independent.


In [ ]:
empirical_rows = []
obs = sensitivity_measurements.set_index("trial_id")
for i in range(5):
    for h_deg in PERTURBATIONS_DEG:
        plus = obs.loc[f"J{i+1}_plus_{h_deg:g}"]
        minus = obs.loc[f"J{i+1}_minus_{h_deg:g}"]
        pp,Rp = measured_pose(plus); pm,Rm = measured_pose(minus)
        denom = 2*np.deg2rad(h_deg)
        jp = (pp-pm)/denom if pp is not None and pm is not None else np.full(3,np.nan)
        jw = rotation_log(Rp @ Rm.T)/denom if Rp is not None and Rm is not None else np.full(3,np.nan)
        out = {"joint_id":i+1,"joint":JOINT_NAMES[i],"step_deg":h_deg}
        for prefix,v in [("empirical_p",jp),("predicted_p",Jservo[:3,i])]:
            out.update({f"{prefix}_{k}_mm_per_rad":x for k,x in zip("xyz",v)})
        for prefix,v in [("empirical_w",jw),("predicted_w",Jservo[3:,i])]:
            out.update({f"{prefix}_{k}_rad_per_rad":x for k,x in zip("xyz",v)})
        out["column_position_error_mm_per_rad"] = np.linalg.norm(jp-Jservo[:3,i])
        out["column_orientation_error_rad_per_rad"] = np.linalg.norm(jw-Jservo[3:,i])
        empirical_rows.append(out)
empirical_jacobian = pd.DataFrame(empirical_rows)
display(empirical_jacobian)


In [ ]:
if plt is not None:
    fig,axs = plt.subplots(1,2,figsize=(12,4))
    for i in range(1,6):
        d = sensitivity_comparison.query("joint_id == @i and delta_servo_deg > 0")
        axs[0].plot(d.delta_servo_deg,d.linearization_position_error_mm,"o-",label=JOINT_NAMES[i-1])
    axs[0].set(xlabel="Individual servo perturbation (deg)",ylabel="FK minus Jacobian change norm (mm)",
               title="Model linearization error")
    axs[0].legend(fontsize=8)
    d = sensitivity_comparison.dropna(subset=["measured_vs_fk_position_error_mm"])
    if len(d):
        axs[1].plot(d.trial_id,d.measured_vs_fk_position_error_mm,"o",label="Measured vs FK")
        axs[1].plot(d.trial_id,d.measured_vs_J_position_error_mm,"x",label="Measured vs Jacobian")
        axs[1].tick_params(axis="x",rotation=90); axs[1].legend()
        axs[1].set(ylabel="Change residual (mm)",title="Physical sensitivity comparison")
    else:
        axs[1].text(.5,.5,"Awaiting sensitivity measurements",ha="center",va="center",transform=axs[1].transAxes)
        axs[1].set_axis_off()
    fig.tight_layout(); fig.savefig(OUTPUT_DIR/"sensitivity_comparison.png",dpi=160); plt.show()


## 8. Explain the observations
Complete the discussion after measurements; the model tables alone are not hardware validation.

- Which of the five poses had the largest position and orientation errors? Compare error magnitudes with instrument uncertainty and repeatability.
- Which individual joint produced the largest displacement? Relate that to the lever arm in `a × (p − o)` and the corresponding Jacobian column.
- Does halving a small step reduce nonlinear-versus-linear position error roughly by a factor of four where the quadratic term is nonzero? Explain noise, zero-motion columns, and higher-order effects.
- Does measured change agree more closely with nonlinear FK or the Jacobian approximation? Discuss geometry, zeros/gains, servo quantization, backlash, compliance, and load.
- Compare measured centered-difference columns at 1° and 2°. Which are consistent within uncertainty? Very small steps may be dominated by measurement noise/deadband.
- For an on-axis tool point, S5 roll has no positional effect, but it does change orientation. An off-axis tool offset changes this result.
- Use the position singular values/rank to discuss configurations with weak Cartesian directions. Do not claim independent control of all six pose coordinates with five positioning joints.

**Before submission:** verify geometry/calibration, collect at least five full poses, include measurement methods and units, export error tables/figures, and (6000 level) include baseline/perturbation observations and Jacobian interpretation. State any orientation components you could not measure.


## 9. Save results and optionally use Google Drive in Colab
The notebook itself is saved in the team's Drive Code folder. This cell exports the current session's tables and parameter snapshot to `OUTPUT_DIR`. Download that folder's files from Colab's Files panel. To write results directly into mounted Drive, mount Drive and set `OUTPUT_DIR` to the exact existing folder path; rerun the export cell below. Do not assume a shared folder appears under MyDrive: add a Drive shortcut if needed, then use its actual mounted path.

Optional Colab setup (run in a separate cell):
```python
from google.colab import drive
drive.mount('/content/drive')
# Inspect the mounted folders, then choose the real destination:
# OUTPUT_DIR = Path('/content/drive/MyDrive/.../Code/dofbot_lab_results')
```
When exporting to a different folder, rerun the plotting cells to save figures there too. Save an executed copy of the notebook after entering your measurements.


In [ ]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
for name,df in {
    "five_test_plan":test_plan, "fk_measurements_current":fk_measurements,
    "fk_comparison":fk_comparison, "sensitivity_plan":sensitivity_plan,
    "sensitivity_measurements_current":sensitivity_measurements,
    "sensitivity_comparison":sensitivity_comparison, "empirical_jacobian":empirical_jacobian,
}.items():
    df.to_csv(OUTPUT_DIR/(name+".csv"),index=False)
parameters = dict(geometry_mm=GEOMETRY, tool_offset_mm=TOOL_OFFSET_MM.tolist(),
    joint_points_home_mm=JOINT_POINTS_HOME.tolist(), axes_home=AXES_HOME.tolist(),
    M_home=M_HOME.tolist(),servo_zero_deg=SERVO_ZERO_DEG.tolist(),servo_gain=SERVO_GAIN.tolist(),
    joint_offset_rad=JOINT_OFFSET_RAD.tolist(),servo_limits_deg=SERVO_LIMITS_DEG.tolist(),
    geometry_verified=GEOMETRY_VERIFIED,calibration_verified=CALIBRATION_VERIFIED,
    gripper_deg=GRIPPER_DEG,baseline_servo_deg=BASELINE_SERVO_DEG.tolist(),
    perturbations_deg=PERTURBATIONS_DEG,position_units="mm",angle_units="radians in model; degrees in servo CSVs",
    orientation_convention="Rz(yaw) Ry(pitch) Rx(roll), base-to-tool",
    measurement_status="Real measurements must be supplied by the experimenter.")
(OUTPUT_DIR/"parameters.json").write_text(json.dumps(parameters,indent=2))
print("Saved tables and parameters to:",OUTPUT_DIR.resolve())


## 10. Optional DOFBOT onboard control helpers
These helpers use the existing vendor `Arm_Lib` on the arm's computer. Cloud Colab cannot drive that board. **Run all does not move the arm.** To use hardware, verify geometry/calibration and local travel limits, set `ENABLE_HARDWARE = True` here, then explicitly call `connect_arm()` and `move_servo_pose(...)` from a separate cell. Commands are rounded to the integer degrees used for this experiment. The preset perturbations therefore use whole degrees.

Position the arm only after checking the pose and the path on your assembly. The range check is not collision checking. Keep the marked gripper reference point and S6 setting unchanged. `read_servo_angles()` is feedback for settling/command checks, not an end-effector pose measurement.


In [ ]:
ENABLE_HARDWARE = False
_arm = None

def connect_arm():
    global _arm
    if not ENABLE_HARDWARE: raise RuntimeError("Hardware disabled. Use the arm's onboard runtime to enable it.")
    if not (GEOMETRY_VERIFIED and CALIBRATION_VERIFIED):
        raise RuntimeError("Verify and flag geometry and calibration first.")
    from Arm_Lib import Arm_Device
    _arm = Arm_Device()
    time.sleep(0.1)
    return _arm

def move_servo_pose(s_deg, duration_ms=1500, settle_s=1.0):
    if not ENABLE_HARDWARE or _arm is None: raise RuntimeError("Call connect_arm() on the local robot first.")
    s = check_servo_limits(s_deg)
    if not np.allclose(s,np.round(s)): raise ValueError("Use whole-degree commands for this lab.")
    if not (0 <= GRIPPER_DEG <= 180) or GRIPPER_DEG != int(GRIPPER_DEG):
        raise ValueError("S6 must be an integer degree in 0..180.")
    if not np.isfinite(duration_ms) or duration_ms < 500 or duration_ms > 10000:
        raise ValueError("Choose duration_ms between 500 and 10000 for deliberate lab motion.")
    if not np.isfinite(settle_s) or settle_s < 0: raise ValueError("Nonnegative settling time required.")
    _arm.Arm_serial_servo_write6(*[int(v) for v in np.round(s)],int(GRIPPER_DEG),int(duration_ms))
    time.sleep(duration_ms/1000 + settle_s)
    return s.copy()  # command only; SDK write does not prove the move succeeded

def read_servo_angles():
    if not ENABLE_HARDWARE or _arm is None: raise RuntimeError("Connect the local arm first.")
    values = []
    for i in range(1,6):
        value = _arm.Arm_serial_servo_read(i)
        if value is None: raise RuntimeError(f"No valid feedback from S{i}; retry/check hardware.")
        values.append(float(value)); time.sleep(0.02)
    return check_servo_limits(values)

# Example manual calls on the robot ONLY, after you have enabled and reviewed the setup:
# connect_arm()
# move_servo_pose(test_sets[0])
# actual_angles = read_servo_angles()
# Measure tool pose independently; enter it in the measurement sheet.
print("Hardware helpers defined; no connection or movement requested.")


## Sources and model scope
Vendor references checked when preparing this notebook (October 5, 2026):

- [Yahboom: single-servo API](https://www.yahboom.net/public/upload/upload-html/1715841824/Control%20a%20single%20servo.html) documents servo IDs, degree ranges, and movement-time behavior.
- [Yahboom: six-servo API](https://www.yahboom.net/public/upload/upload-html/1775560275/05.Control%20all%20servo.html) documents `Arm_serial_servo_write6(S1, S2, S3, S4, S5, S6, time)` and the S5 0–270° range.
- [Yahboom official DOFBOT repository](https://github.com/YahboomTechnology/dofbot-Pi) provides hardware and programming material. Use the documentation for your exact DOFBOT variant and installed controller library.

The editable model is an ideal rigid serial chain, not a manufacturer-certified geometry. Parameter verification and physical pose observations are required for a hardware result. The analytic/numerical checks demonstrate mathematical consistency only.
